# Synthetic Philadelphia - Reworked Pipeline

This version caches the fully rasterized city and parallelizes the per-agent workflow: destination diary generation, dense trajectory generation, sparse sampling, and dataframe extraction.

In [12]:
from pathlib import Path
import json
import time

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from joblib import Parallel, cpu_count, delayed
from shapely.geometry import box

import nomad.map_utils as nm
from nomad.city_gen import RasterCity, load as load_city_pickle, save as save_city_pickle
from nomad.io.base import from_df, to_file
from nomad.traj_gen import Agent, Population

## Configuration

In [13]:
LARGE_BOX = box(-75.1905, 39.9235, -75.1425, 39.9535)
MEDIUM_BOX = box(-75.1665, 39.9385, -75.1425, 39.9535)

USE_FULL_CITY = False
OUTPUT_DIR = Path("output_rework")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ox.settings.cache_folder = "cache_rework"

if USE_FULL_CITY:
    BOX_NAME = "full"
    POLY = "Philadelphia, Pennsylvania, USA"
else:
    BOX_NAME = "large"
    POLY = LARGE_BOX

SPATIAL_GPKG = OUTPUT_DIR / f"spatial_data_{BOX_NAME}.gpkg"
ROTATION_METADATA_PATH = OUTPUT_DIR / f"rotation_metadata_{BOX_NAME}.json"
CITY_CACHE_PICKLE = OUTPUT_DIR / f"raster_city_{BOX_NAME}.pkl"

REGENERATE_SPATIAL_DATA = False
REBUILD_CITY_CACHE = False
N_JOBS = -1

config = {
    "box_name": BOX_NAME,
    "block_side_length": 15.0,
    "hub_size": 100,
    "N": 200,
    "name_seed": 42,
    "name_count": 2,
    "epr_params": {
        "datetime": "2025-05-23 00:00-05:00",
        "end_time": "2025-07-01 00:00-05:00",
        "epr_time_res": 15,
        "rho": 0.4,
        "gamma": 0.3,
        "seed_base": 100,
    },
    "traj_params": {
        "dt": 0.5,
        "seed_base": 200,
    },
    "sampling_params": {
        "beta_ping": 7,
        "beta_start": 300,
        "beta_durations": 55,
        "ha": 11.5 / 15,
        "seed_base": 1,
    },
}

## Helpers

In [14]:
def section(title):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)


def write_dataset(df, path, partition_cols):
    table = pa.Table.from_pandas(df, preserve_index=False)
    ds.write_dataset(
        table,
        base_dir=str(path),
        format="parquet",
        partitioning=partition_cols,
        existing_data_behavior="delete_matching",
    )


def save_traj_dataset(df, path):
    out = from_df(df, mixed_timezone_behavior="naive")
    to_file(
        out,
        path=path,
        format="parquet",
        partition_by=["date"],
        existing_data_behavior="delete_matching",
    )

## Spatial Data Cache

In [15]:
section("SPATIAL DATA")

if REGENERATE_SPATIAL_DATA or not SPATIAL_GPKG.exists():
    t0 = time.time()
    buildings = nm.download_osm_buildings(
        POLY,
        crs="EPSG:3857",
        schema="garden_city",
        clip=True,
        infer_building_types=True,
        explode=True,
    )
    print(f"Buildings download: {time.time() - t0:>6.2f}s ({len(buildings):,} buildings)")

    if USE_FULL_CITY:
        boundary_polygon = nm.get_city_boundary_osm(POLY, simplify=True)[0]
        boundary_polygon = gpd.GeoSeries([boundary_polygon], crs="EPSG:4326").to_crs("EPSG:3857").iloc[0]
    else:
        boundary_polygon = gpd.GeoDataFrame(geometry=[POLY], crs="EPSG:4326").to_crs("EPSG:3857").geometry.iloc[0]

    buildings = gpd.clip(buildings, gpd.GeoDataFrame(geometry=[boundary_polygon], crs="EPSG:3857"))
    buildings = nm.remove_overlaps(buildings).reset_index(drop=True)

    t1 = time.time()
    streets = nm.download_osm_streets(
        POLY,
        crs="EPSG:3857",
        clip=True,
        explode=True,
        graphml_path=OUTPUT_DIR / "streets_consolidated.graphml",
    ).reset_index(drop=True)
    print(f"Streets download:   {time.time() - t1:>6.2f}s ({len(streets):,} streets)")

    t2 = time.time()
    rotated_streets, rotation_deg = nm.rotate_streets_to_align(streets, k=200)
    all_streets = streets.geometry.union_all()
    rotation_origin = (all_streets.centroid.x, all_streets.centroid.y)
    rotated_buildings = nm.rotate(buildings, rotation_deg=rotation_deg, origin=rotation_origin)
    rotated_boundary = nm.rotate(
        gpd.GeoDataFrame(geometry=[boundary_polygon], crs="EPSG:3857"),
        rotation_deg=rotation_deg,
        origin=rotation_origin,
    )
    print(f"Grid rotation:      {time.time() - t2:>6.2f}s ({rotation_deg:.2f} deg)")

    if SPATIAL_GPKG.exists():
        SPATIAL_GPKG.unlink()
    rotated_buildings.to_file(SPATIAL_GPKG, layer="buildings", driver="GPKG")
    rotated_streets.to_file(SPATIAL_GPKG, layer="streets", driver="GPKG", mode="a")
    rotated_boundary.to_file(SPATIAL_GPKG, layer="boundary", driver="GPKG", mode="a")

    with open(ROTATION_METADATA_PATH, "w") as f:
        json.dump({"rotation_deg": rotation_deg, "rotation_origin": rotation_origin}, f)
else:
    print(f"Loading spatial data from {SPATIAL_GPKG}")

with open(ROTATION_METADATA_PATH) as f:
    rotation_metadata = json.load(f)

rotation_deg = rotation_metadata["rotation_deg"]
rotation_origin = tuple(rotation_metadata["rotation_origin"])


SPATIAL DATA
Buildings download:  23.62s (53,935 buildings)
Streets download:    46.43s (5,221 streets)
Grid rotation:        0.48s (9.43 deg)


## Raster City Cache

In [16]:
section("RASTER CITY CACHE")

if REBUILD_CITY_CACHE or not CITY_CACHE_PICKLE.exists():
    buildings = gpd.read_file(SPATIAL_GPKG, layer="buildings")
    streets = gpd.read_file(SPATIAL_GPKG, layer="streets")
    boundary = gpd.read_file(SPATIAL_GPKG, layer="boundary")

    t0 = time.time()
    city_for_cache = RasterCity(
        boundary.geometry.iloc[0],
        streets,
        buildings,
        block_side_length=config["block_side_length"],
        resolve_overlaps=True,
        other_building_behavior="filter",
        rotation_deg=rotation_deg,
        rotation_origin=rotation_origin,
    )
    print(f"City generation:    {time.time() - t0:>6.2f}s")

    t1 = time.time()
    city_for_cache.get_street_graph()
    print(f"Street graph:       {time.time() - t1:>6.2f}s")

    t2 = time.time()
    city_for_cache._build_hub_network(hub_size=config["hub_size"])
    print(f"Hub network:        {time.time() - t2:>6.2f}s")

    save_city_pickle(city_for_cache, CITY_CACHE_PICKLE)
    print(f"Saved raster city cache to {CITY_CACHE_PICKLE}")
else:
    print(f"Loading raster city cache from {CITY_CACHE_PICKLE}")


RASTER CITY CACHE
Generated 104,177 blocks (in 0.86s)
Assigning block types...
Block types assigned (in 0.04s)
Assigning streets...
Verifying street connectivity...
  Streets: 28,086 kept, 0 discarded (in 0.07s)
Adding buildings to city...
  Added 19981 buildings, skipped 23637 due to overlap (adding took 19.23s)
City generation:     23.29s
Street graph:         0.07s
Hub network:          0.78s
Saved raster city cache to output_rework/raster_city_large.pkl


In [17]:
def load_city_from_cache():
    city = load_city_pickle(CITY_CACHE_PICKLE)
    city.rotation_deg = rotation_metadata["rotation_deg"]
    city.rotation_origin = tuple(rotation_metadata["rotation_origin"])
    city.compute_gravity(exponent=2.0, callable_only=True)
    city.compute_shortest_paths(callable_only=True)
    return city


city = load_city_from_cache()

summary_df = pd.DataFrame({
    "component": ["blocks", "streets", "buildings", "hubs", "nearby_door_pairs"],
    "count": [
        len(city.blocks_gdf),
        len(city.streets_gdf),
        len(city.buildings_gdf),
        len(city.hubs),
        len(city.mh_dist_nearby_doors),
    ],
})
print(summary_df.to_string(index=False))
print(city.buildings_gdf.building_type.value_counts())

        component   count
           blocks  104177
          streets   28086
        buildings   19981
             hubs     100
nearby_door_pairs 1003389
building_type
home         19325
retail         438
workplace      129
park            89
Name: count, dtype: int64


## Agent Parameters

In [18]:
section("AGENT PARAMETERS")

population_template = Population(city)
population_template.generate_agents(
    N=config["N"],
    seed=config["name_seed"],
    name_count=config["name_count"],
    datetimes=config["epr_params"]["datetime"],
)

agents = list(population_template.roster.values())

agent_params = pd.DataFrame({
    "identifier": [agent.identifier for agent in agents],
    "home": [agent.home for agent in agents],
    "workplace": [agent.workplace for agent in agents],
    "datetime": [config["epr_params"]["datetime"]] * len(agents),
    "dest_seed": config["epr_params"]["seed_base"] + np.arange(len(agents)),
    "traj_seed": config["traj_params"]["seed_base"] + np.arange(len(agents)),
    "sampling_seed": config["sampling_params"]["seed_base"] + np.arange(len(agents)),
})

agent_params["beta_start"] = config["sampling_params"]["beta_start"]
agent_params["beta_ping"] = config["sampling_params"]["beta_ping"]
agent_params["beta_durations"] = config["sampling_params"]["beta_durations"]
agent_params["ha"] = config["sampling_params"]["ha"]

agent_params_path = OUTPUT_DIR / f"agent_params_{BOX_NAME}.parquet"
agent_params.to_parquet(agent_params_path, index=False)
agent_params.head()


AGENT PARAMETERS


,identifier,home,workplace,datetime,dest_seed,traj_seed,sampling_seed,beta_start,beta_ping,beta_durations,ha
0,clever_colden,h-x84-y33,w-x271-y255,2025-05-23 00:00-05:00,100,200,1,300,7,55,0.766667
1,clever_noether,h-x244-y246,w-x246-y280,2025-05-23 00:00-05:00,101,201,2,300,7,55,0.766667
2,cocky_clarke,h-x322-y127,w-x201-y209,2025-05-23 00:00-05:00,102,202,3,300,7,55,0.766667
3,cocky_panini,h-x156-y51,w-x276-y262,2025-05-23 00:00-05:00,103,203,4,300,7,55,0.766667
4,compassionate_davinci,h-x138-y180,w-x234-y129,2025-05-23 00:00-05:00,104,204,5,300,7,55,0.766667


## Per-Agent Generation

In [19]:
end_time = pd.Timestamp(config["epr_params"]["end_time"])


def effective_n_jobs(n_jobs, task_count):
    """Return the number of worker batches for the configured joblib setting."""
    if n_jobs < 0:
        n_jobs = max(cpu_count() + 1 + n_jobs, 1)
    return min(n_jobs, task_count)


def simulate_agent(row, city_worker):
    agent = Agent(
        identifier=row.identifier,
        city=city_worker,
        home=row.home,
        workplace=row.workplace,
        datetime=row.datetime,
    )
    agent.generate_dest_diary(
        end_time=end_time,
        epr_time_res=config["epr_params"]["epr_time_res"],
        rho=config["epr_params"]["rho"],
        gamma=config["epr_params"]["gamma"],
        seed=int(row.dest_seed),
    )
    dest_diary = agent.destination_diary.copy()
    dest_diary["identifier"] = row.identifier

    agent.generate_trajectory(
        dt=config["traj_params"]["dt"],
        seed=int(row.traj_seed),
    )
    full_points = len(agent.trajectory)
    agent.sample_trajectory(
        beta_start=float(row.beta_start),
        beta_durations=float(row.beta_durations),
        beta_ping=float(row.beta_ping),
        ha=float(row.ha),
        seed=int(row.sampling_seed),
        replace_sparse_traj=True,
        cache_traj=True,
    )

    sparse = agent.sparse_traj.reset_index(drop=True).copy()
    diary = agent.diary.copy()
    summary = {
        "user_id": row.identifier,
        "dest_entries": len(dest_diary),
        "diary_entries": len(diary),
        "full_points": full_points,
        "sparse_points": len(sparse),
        "beta_start": row.beta_start,
        "beta_ping": row.beta_ping,
        "beta_durations": row.beta_durations,
        "ha": row.ha,
    }
    return sparse, diary, dest_diary, summary


def simulate_agent_batch(agent_batch):
    """Simulate one dataframe chunk using one loaded city per worker task."""
    city_worker = load_city_from_cache()
    return [simulate_agent(row, city_worker) for row in agent_batch.itertuples(index=False)]

In [20]:
section("PARALLEL AGENT GENERATION")

n_workers = effective_n_jobs(N_JOBS, len(agent_params))
agent_batches = [agent_params.iloc[idx] for idx in np.array_split(np.arange(len(agent_params)), n_workers)]
t0 = time.time()
batch_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(simulate_agent_batch)(batch) for batch in agent_batches
)
generation_time = time.time() - t0

results = [result for batch in batch_results for result in batch]
sparse_parts, diary_parts, dest_diary_parts, summary_rows = zip(*results)
sparse_df = pd.concat(sparse_parts, ignore_index=True)
diaries_df = pd.concat(diary_parts, ignore_index=True)
dest_diaries_df = pd.concat(dest_diary_parts, ignore_index=True)
generation_summary = pd.DataFrame(summary_rows)

print(f"Generated {len(agent_params)} agents in {generation_time:.2f}s")
print(generation_summary[["full_points", "sparse_points", "dest_entries", "diary_entries"]].sum())
generation_summary.head()


PARALLEL AGENT GENERATION


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  4.4min remaining: 28.8min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:  4.5min remaining: 12.3min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  4.5min remaining:  6.7min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  4.5min remaining:  3.9min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  4.6min remaining:  2.3min
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  4.6min remaining:  1.2min


Generated 200 agents in 283.03s
full_points      22464200
sparse_points      239023
dest_entries        79806
diary_entries       91725
dtype: int64


[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  4.7min finished


,user_id,dest_entries,diary_entries,full_points,sparse_points,beta_start,beta_ping,beta_durations,ha
0,clever_colden,379,409,112321,1138,300,7,55,0.766667
1,clever_noether,429,533,112321,1067,300,7,55,0.766667
2,cocky_clarke,393,476,112321,986,300,7,55,0.766667
3,cocky_panini,411,519,112321,1078,300,7,55,0.766667
4,compassionate_davinci,432,499,112321,1134,300,7,55,0.766667


## Reproject And Save

In [21]:
section("REPROJECT AND SAVE")

poi_data = pd.DataFrame({
    "building_id": city.buildings_gdf["id"].values,
    "x": (city.buildings_gdf["door_cell_x"].astype(float) + 0.5).values,
    "y": (city.buildings_gdf["door_cell_y"].astype(float) + 0.5).values,
})

sparse_df = city.to_mercator(sparse_df)
diaries_df = diaries_df.merge(poi_data, left_on="location", right_on="building_id", how="left")
diaries_df = diaries_df.drop(columns=["building_id"])
diaries_df = city.to_mercator(diaries_df)

sparse_df["date"] = pd.to_datetime(sparse_df["timestamp"], unit="s").dt.date.astype(str)
diaries_df["date"] = pd.to_datetime(diaries_df["timestamp"], unit="s").dt.date.astype(str)
dest_diaries_df["date"] = pd.to_datetime(dest_diaries_df["datetime"]).dt.date.astype(str)
generation_summary["date"] = pd.to_datetime(config["epr_params"]["datetime"]).date().isoformat()

config_path = OUTPUT_DIR / f"config_{BOX_NAME}.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

save_traj_dataset(sparse_df, OUTPUT_DIR / f"sparse_traj_{BOX_NAME}")
save_traj_dataset(diaries_df, OUTPUT_DIR / f"diaries_{BOX_NAME}")
write_dataset(dest_diaries_df, OUTPUT_DIR / f"dest_diaries_{BOX_NAME}", ["date"])
write_dataset(generation_summary, OUTPUT_DIR / f"homes_{BOX_NAME}", ["date"])

write_dataset(sparse_df, OUTPUT_DIR / "device_level", ["date"])
write_dataset(diaries_df, OUTPUT_DIR / "travel_diaries", ["date"])

print(f"Config saved to {config_path}")
print(f"Agent params saved to {agent_params_path}")
print(f"Sparse trajectories: {OUTPUT_DIR / f'sparse_traj_{BOX_NAME}'}")
print(f"Realized diaries:    {OUTPUT_DIR / f'diaries_{BOX_NAME}'}")
print(f"Destination diaries: {OUTPUT_DIR / f'dest_diaries_{BOX_NAME}'}")


REPROJECT AND SAVE
Config saved to output_rework/config_large.json
Agent params saved to output_rework/agent_params_large.parquet
Sparse trajectories: output_rework/sparse_traj_large
Realized diaries:    output_rework/diaries_large
Destination diaries: output_rework/dest_diaries_large


## Quick Checks

In [22]:
print("Sparse rows:", len(sparse_df))
print("Diary rows:", len(diaries_df))
print("Destination diary rows:", len(dest_diaries_df))
print("Sparse ratio:", f"{len(sparse_df) / generation_summary['full_points'].sum():.2%}")
generation_summary.describe()

Sparse rows: 239023
Diary rows: 91725
Destination diary rows: 79806
Sparse ratio: 1.06%


,dest_entries,diary_entries,full_points,sparse_points,beta_start,beta_ping,beta_durations,ha
count,200.00000,200.000000,200.0,200.000000,200.0,200.0,200.0,2.000000e+02
mean,399.03000,458.625000,112321.0,1195.115000,300.0,7.0,55.0,7.666667e-01
std,22.45731,40.996315,0.0,122.015164,0.0,0.0,0.0,1.113009e-16
min,319.00000,365.000000,112321.0,833.000000,300.0,7.0,55.0,7.666667e-01
25%,384.00000,431.000000,112321.0,1103.750000,300.0,7.0,55.0,7.666667e-01
50%,399.00000,453.000000,112321.0,1179.000000,300.0,7.0,55.0,7.666667e-01
75%,412.00000,483.000000,112321.0,1282.750000,300.0,7.0,55.0,7.666667e-01
max,459.00000,599.000000,112321.0,1523.000000,300.0,7.0,55.0,7.666667e-01
